<a href="https://colab.research.google.com/github/kingdragonlord/ChatDev/blob/main/FinBERT_Sentiment_Enrichment(RunAll).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================================================================
# CELL 1: SETUP & GPU VERIFICATION
# ===================================================================
!pip install transformers torch pandas --quiet
import torch
import pandas as pd
import os
from tqdm.auto import tqdm

print("✅ Libraries installed and imported.")

# --- Verify GPU is available ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU is available. Using device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ GPU not available. Using CPU. This will be VERY slow.")

# Register tqdm with pandas for progress bars on .apply()
tqdm.pandas()

In [ ]:
# ===================================================================
# CELL 2: LOAD RAW HEADLINES DATA
# ===================================================================

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted.")

# --- Load the Consolidated Raw Headline Data ---
GDRIVE_PROJECT_BASE = "/content/drive/MyDrive/TFT_Project_Final"
RAW_DATA_PATH = os.path.join(GDRIVE_PROJECT_BASE, "data_cache/alpaca_raw_headlines.parquet")
# --- Define the final output path ---
FINAL_OUTPUT_PATH = os.path.join(GDRIVE_PROJECT_BASE, "data_cache/alpaca_news_features_finbert.parquet")

try:
    df = pd.read_parquet(RAW_DATA_PATH)
    print(f"✅ Successfully loaded the raw headlines dataset.")
    print(f"   -> Found {len(df):,} total headlines to process.")

    # For development, you can uncomment the line below to work with a smaller sample
    # df = df.sample(10000, random_state=42).copy()

except FileNotFoundError:
    print(f"🛑 CRITICAL: File not found at '{RAW_DATA_PATH}'.")
    print("   Please ensure the Alpaca scraper ran and was consolidated successfully.")
    df = None

In [ ]:
# ===================================================================
# CELL 3: INITIALIZE FinBERT PIPELINE
# ===================================================================
from transformers import pipeline

if df is not None:
    print("--- Initializing FinBERT sentiment analysis pipeline ---")
    print("This will download the model from Hugging Face (approx. 400MB)...")

    # Use the pipeline for simplicity and batching efficiency.
    # We pass the device index (0 for the first GPU) to ensure it runs on the GPU.
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model="ProsusAI/finbert",
        device=0 if torch.cuda.is_available() else -1
    )

    print("✅ FinBERT pipeline ready.")

In [ ]:
# ===================================================================
# CELL 4: SENTIMENT SCORING (THE MAIN EVENT)
# ===================================================================
import numpy as np

if df is not None:
    # --- Define a function to process headlines in batches ---
    def get_sentiment_scores(headlines, batch_size=64):
        all_scores = []
        # Use tqdm to show progress for the batches
        for i in tqdm(range(0, len(headlines), batch_size), desc="Processing Headline Batches"):
            batch = headlines[i : i + batch_size]
            # The pipeline processes the whole batch at once on the GPU
            results = sentiment_pipeline(batch)
            all_scores.extend(results)
        return all_scores

    # --- Run the sentiment analysis ---
    print("\n--- Starting sentiment analysis on all headlines ---")
    print(f"Processing {len(df):,} headlines. This will take time, even with a GPU.")

    # Get the raw sentiment results (list of dictionaries)
    headline_sentiments = get_sentiment_scores(df['headline'].tolist())

    # --- Process the results into a numerical score ---
    score_map = {'positive': 1.0, 'negative': -1.0, 'neutral': 0.0}

    # Extract the label and score, then calculate the final numerical score
    df['finbert_label'] = [res['label'] for res in headline_sentiments]
    df['finbert_confidence'] = [res['score'] for res in headline_sentiments]

    # The final score is the direction (+1, -1, 0) multiplied by the model's confidence
    df['finbert_score'] = df.apply(lambda row: score_map[row['finbert_label']] * row['finbert_confidence'], axis=1)

    print("\n--- ✅ Sentiment scoring complete! ---")
    print("\n--- Sample of scored headlines: ---")
    display(df[['headline', 'finbert_label', 'finbert_score']].sample(10))

In [ ]:
# ===================================================================
# CELL 5: AGGREGATE SCORES & CREATE FINAL FEATURES
# ===================================================================
if df is not None:
    print("--- Aggregating daily scores and engineering final features ---")

    # Ensure date is just the date part for daily grouping
    df['date_only'] = df['date'].dt.date

    # Group by symbol and date, then aggregate
    daily_summary = df.groupby(['symbol', 'date_only']).agg(
        # Sum of all sentiment scores for the day
        news_sentiment_score_24h=('finbert_score', 'sum'),
        # Count of all news stories for the day
        news_story_count_24h=('headline', 'count')
    ).reset_index()

    # Rename date_only back to date for consistency
    daily_summary.rename(columns={'date_only': 'date'}, inplace=True)
    daily_summary['date'] = pd.to_datetime(daily_summary['date'], utc=True)

    # --- Engineer the 'has_major_news' feature ---
    # A day has major news if the story count is an outlier (> 2 std devs from the rolling 20-day mean)
    # OR if the absolute sentiment score is an outlier.

    # Calculate rolling stats per symbol
    daily_summary = daily_summary.sort_values(by=['symbol', 'date'])
    count_avg = daily_summary.groupby('symbol')['news_story_count_24h'].transform(lambda x: x.rolling(20, 1).mean())
    count_std = daily_summary.groupby('symbol')['news_story_count_24h'].transform(lambda x: x.rolling(20, 1).std())
    sentiment_avg = daily_summary.groupby('symbol')['news_sentiment_score_24h'].transform(lambda x: x.rolling(20, 1).mean())
    sentiment_std = daily_summary.groupby('symbol')['news_sentiment_score_24h'].transform(lambda x: x.rolling(20, 1).std())

    is_high_volume = daily_summary['news_story_count_24h'] > (count_avg + 2 * count_std)
    is_high_sentiment = abs(daily_summary['news_sentiment_score_24h']) > (abs(sentiment_avg) + 2 * sentiment_std)

    daily_summary['has_major_news'] = (is_high_volume | is_high_sentiment).astype(int)

    # --- Finalize the DataFrame ---
    final_features_df = daily_summary[['date', 'symbol', 'news_story_count_24h', 'news_sentiment_score_24h', 'has_major_news']]

    print("\n--- Saving final feature-engineered dataset ---")
    final_features_df.to_parquet(FINAL_OUTPUT_PATH, index=False)

    print(f"\n✅ Successfully created and saved the final news features.")
    print(f"   -> File saved to: {FINAL_OUTPUT_PATH}")
    print("\n--- Final Data Sample: ---")
    display(final_features_df.sample(10))

In [ ]:
# ===================================================================
# FINAL VERIFICATION CELL
# ===================================================================
import pandas as pd
import os

print("--- Running Post-Processing Verification ---")

# --- Configuration ---
# This path should match the output path defined in the notebook
FINAL_FILE_PATH = "/content/drive/MyDrive/TFT_Project_Final/data_cache/alpaca_news_features_finbert.parquet"
# This should match the end date of your raw headline scrape
EXPECTED_END_DATE = pd.to_datetime("2025-08-11", utc=True)

# --- 1. File Existence and Loading Check ---
if not os.path.exists(FINAL_FILE_PATH):
    print(f"🛑 FAILED: Final feature file not found at {FINAL_FILE_PATH}")
else:
    print(f"✅ Found final file: {FINAL_FILE_PATH}")
    try:
        df = pd.read_parquet(FINAL_FILE_PATH)
        print(f"  - Successfully loaded file with {len(df):,} rows.")

        # --- 2. Date Range Verification ---
        latest_date = df['date'].max()
        print(f"\n--- Date Range ---")
        print(f"  - Earliest feature date: {df['date'].min().date()}")
        print(f"  - Latest feature date:   {latest_date.date()}")

        if (EXPECTED_END_DATE - latest_date).days <= 3:
            print(f"  - ✅ SUCCESS: The latest date is current.")
        else:
            print(f"  - 🛑 WARNING: The latest date ({latest_date.date()}) is earlier than expected ({EXPECTED_END_DATE.date()}).")

        # --- 3. NaN Value Check ---
        nan_counts = df.isnull().sum()
        nan_issues = nan_counts[nan_counts > 0]
        if nan_issues.empty:
            print("\n--- NaN Check ---")
            print("  - ✅ SUCCESS: No NaN values found in the final feature set.")
        else:
            print("\n--- NaN Check ---")
            print("  - 🛑 FAILED: Found NaN values in the final feature set:")
            print(nan_issues)

    except Exception as e:
        print(f"🛑 FAILED: Could not load or inspect the parquet file. Error: {e}")

print("\n--- Verification Complete ---")

#EDA

In [ ]:
from matplotlib import pyplot as plt
df_8['news_sentiment_score_24h'].plot(kind='line', figsize=(8, 4), title='news_sentiment_score_24h')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
_df_7['news_story_count_24h'].plot(kind='line', figsize=(8, 4), title='news_story_count_24h')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  xs = series['date']
  ys = series['news_story_count_24h']

  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_6.sort_values('date', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('date')
_ = plt.ylabel('news_story_count_24h')

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['date']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'date'}, axis=1)
              .sort_values('date', ascending=True))
  xs = counted['date']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_5.sort_values('date', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('date')
_ = plt.ylabel('count()')

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  xs = series['date']
  ys = series['news_sentiment_score_24h']

  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_4.sort_values('date', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('date')
_ = plt.ylabel('news_sentiment_score_24h')

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  xs = series['date']
  ys = series['news_story_count_24h']

  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_3.sort_values('date', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('date')
_ = plt.ylabel('news_story_count_24h')

In [ ]:
from matplotlib import pyplot as plt
_df_2.plot(kind='scatter', x='news_story_count_24h', y='news_sentiment_score_24h', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
_df_1['news_sentiment_score_24h'].plot(kind='hist', bins=20, title='news_sentiment_score_24h')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
_df_0['news_story_count_24h'].plot(kind='hist', bins=20, title='news_story_count_24h')
plt.gca().spines[['top', 'right',]].set_visible(False)

#TEST

In [ ]:
# ===================================================================
# CELL: STAND-ALONE FinBERT INFERENCE TEST
# ===================================================================
!pip install transformers torch --quiet
from transformers import pipeline
import torch

print("--- Initializing FinBERT sentiment analysis pipeline ---")
print("This will download the model from Hugging Face (approx. 400MB)...")

# --- Initialize the pipeline ---
# To force CPU, we use device=-1. For a GPU, you would use device=0.
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    device=-1 # Use -1 for CPU, 0 for GPU
)
print("✅ FinBERT pipeline ready.")


# --- Define some realistic, nuanced test headlines ---
test_headlines = [
    "Apple beats earnings expectations but guidance is weak for the next quarter.", # Ambiguous
    "Tesla stock soars after record delivery numbers surprise Wall Street.",       # Clearly Positive
    "Regulators open investigation into a major bank over compliance failures.",   # Clearly Negative
    "The company announced a stock buyback program.",                             # Generally Positive
    "Analysts maintain a 'Hold' rating on the stock."                             # Neutral
]

print("\n--- Running sentiment analysis on test headlines ---")
raw_results = sentiment_pipeline(test_headlines)

print("\n\n--- 🕵️‍♂️ INFERENCE RESULTS ---")

# --- Define the logic to convert labels to numbers ---
score_map = {'positive': 1.0, 'negative': -1.0, 'neutral': 0.0}

for i, headline in enumerate(test_headlines):
    result = raw_results[i]
    label = result['label']
    confidence = result['score']

    # This is the final numerical score we will feed to the TFT model
    final_score = score_map[label] * confidence

    print(f"\nHEADLINE {i+1}: \"{headline}\"")
    print(f"  -> FinBERT Raw Output: {{'label': '{label}', 'score': {confidence:.4f}}}")
    print(f"  -> Final Numerical Score: {final_score:.4f}")

#EDA

In [ ]:
# ===================================================================
# CELL 1: SETUP, MOUNT DRIVE & LOAD FINAL DATA
# ===================================================================
!pip install pandas matplotlib seaborn --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("✅ Libraries installed and imported.")
plt.style.use('seaborn-v0_8-whitegrid')

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted.")

# --- Load the Final Feature-Engineered Dataset ---
GDRIVE_PROJECT_BASE = "/content/drive/MyDrive/TFT_Project_Final"
DATA_FILE_PATH = os.path.join(GDRIVE_PROJECT_BASE, "data_cache/alpaca_news_features_finbert.parquet")

try:
    df = pd.read_parquet(DATA_FILE_PATH)
    print(f"✅ Successfully loaded the final FinBERT features dataset.")
    print(f"   -> Found {len(df):,} daily records for {df['symbol'].nunique()} symbols.")
except FileNotFoundError:
    print(f"🛑 CRITICAL: File not found at '{DATA_FILE_PATH}'.")
    df = None

In [ ]:
# ===================================================================
# CELL 2: HIGH-LEVEL STATISTICS & `has_major_news` ANALYSIS
# ===================================================================
if df is not None:
    print("--- Descriptive Statistics for New Features ---")
    # Focus on the newly created features
    display(df[['news_story_count_24h', 'news_sentiment_score_24h']].describe())

    print("\n\n--- Analysis of 'has_major_news' Feature ---")
    major_news_counts = df['has_major_news'].value_counts()

    if 1 in major_news_counts:
        count_of_major_days = major_news_counts[1]
        percentage = (count_of_major_days / len(df)) * 100
        print(f"  - ✅ The 'has_major_news' flag is 1 on {count_of_major_days:,} days.")
        print(f"  - This represents {percentage:.2f}% of the total dataset, which is expected for a sparse feature.")
    else:
        print("  - ⚠️ No days were flagged with 'has_major_news' = 1. The threshold might be too strict.")

    print("\n--- Example of days WITH Major News ---")
    # Show some examples where the flag was triggered
    display(df[df['has_major_news'] == 1].sample(10, random_state=42))

In [ ]:
# ===================================================================
# CELL 3: VISUALIZING FEATURE DISTRIBUTIONS
# ===================================================================
if df is not None:
    fig, axes = plt.subplots(2, 1, figsize=(15, 12))

    # --- Plot 1: Distribution of Daily Story Count ---
    # We clip the count at 50 for better visualization, as there are long tails
    sns.histplot(df['news_story_count_24h'].clip(upper=50), bins=50, ax=axes[0], color='teal')
    axes[0].set_title('Distribution of Daily News Story Count (Clipped at 50)', fontsize=16)
    axes[0].set_xlabel('Number of Headlines in a Day')

    # --- Plot 2: Distribution of Daily Sentiment Score ---
    # We clip the sentiment score to handle extreme outliers
    sns.histplot(df['news_sentiment_score_24h'].clip(-10, 10), bins=100, ax=axes[1], color='darkred', kde=True)
    axes[1].set_title('Distribution of Aggregated Daily Sentiment Score (Clipped at ±10)', fontsize=16)
    axes[1].set_xlabel('Sum of FinBERT Scores in a Day')
    axes[1].axvline(0, color='black', linestyle='--')

    plt.tight_layout()
    plt.show()

In [ ]:
# ===================================================================
# CELL 4: SANITY CHECK - TOP POSITIVE & NEGATIVE DAYS
# ===================================================================
if df is not None:
    # --- Find the days across all stocks with the highest and lowest sentiment scores ---
    df_sorted_sentiment = df.sort_values('news_sentiment_score_24h', ascending=False)

    print("--- Top 10 Most POSITIVE News Days Across All Stocks ---")
    display(df_sorted_sentiment.head(10))

    print("\n\n--- Top 10 Most NEGATIVE News Days Across All Stocks ---")
    display(df_sorted_sentiment.tail(10))

In [ ]:
from google.colab import runtime
runtime.unassign()